# Atividade Avaliativa - Construindo um Pipeline de Dados Robusto

Em exemplos anteriores, você trabalhou com conjuntos de dados limpos e pré-empacotados. Na realidade, os dados raramente são simples assim. Em visão computacional, as imagens frequentemente vêm em tamanhos e formatos diferentes e precisam ser pré-processadas antes que um modelo possa aprender com elas. Lidar com isso manualmente para milhares de imagens seria tanto tedioso quanto sujeito a erros.

Nesta tarefa, você trabalhará com o conjunto de dados [Plants Classification](https://www.kaggle.com/datasets/marquis03/plants-classification), que contém 30.000 imagens `.jpg` distribuídas em 30 espécies de plantas, como aloe vera, banana, espinafre e melancia. Como muitos conjuntos de dados do mundo real, as imagens variam em tamanho e qualidade e estão organizadas em pastas por classe. Para este exercício, você usará um subconjunto de 3.000 imagens.

É aqui que entra um pipeline de dados. Você terá uma experiência prática construindo um conjunto de dados personalizado, aplicando as transformações necessárias e carregando seus dados em lotes (*batches*). Estes são os primeiros passos essenciais antes de treinar um modelo de *deep learning*.

**O Que Você Fará Nesta Tarefa**

* Acessar e explorar a estrutura de um conjunto de dados de imagens.
* Construir uma classe `Dataset` personalizada para carregar suas imagens e rótulos sob demanda.
* Definir uma série de `transformações`, incluindo redimensionamento, conversão para tensor e `normalização`, para pré-processar os dados.
* Definir transformações de aumento de dados (*augmentation*) para aprimorar o conjunto de dados de treinamento.
* Dividir o conjunto de dados em conjuntos de treinamento, validação e teste, aplicando as transformações apropriadas a cada um e criando instâncias de `DataLoader` para um agrupamento em lotes eficiente.

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">DICAS PARA O SUCESSO NA AVALIAÇÃO DA SUA TAREFA:</h4>

* Todas as células estão congeladas, exceto aquelas onde você precisa enviar suas soluções ou quando for explicitamente mencionado que você pode interagir com elas.

* Em cada célula de exercício, procure pelos comentários `### INICIE SEU CÓDIGO AQUI ###` e `### TERMINE SEU CÓDIGO AQUI ###`. Eles mostram onde você deve escrever o código da solução. **Não adicione nem altere nenhum código que esteja fora desses comentários**.

* Você pode adicionar novas células para fazer experimentos, mas elas serão ignoradas pelo corretor automático. Portanto, não dependa de células recém-criadas para hospedar o código da sua solução; utilize os locais fornecidos para isso.

* Evite o uso de variáveis globais, a menos que seja estritamente necessário. O corretor testa seu código em um ambiente isolado, sem executar todas as células desde o topo. Como resultado, as variáveis globais podem estar indisponíveis no momento da correção. As variáveis globais que foram feitas para ser usadas estarão definidas em LETRAS MAIÚSCULAS.

* Para enviar seu notebook para avaliação, primeiro salve-o clicando no ícone 💾 no canto superior esquerdo da página e, em seguida, clique no botão `Submit assignment` (Enviar tarefa) no canto superior direito da página.
---

## Sumário / Índice

* [Importações](#imports)
* [1 - Acesso aos Dados](#1---data-access)
    * [1.1 - Explorando o Conjunto de Dados](#11---exploring-the-dataset)
    * [1.2 - Criando uma Classe Dataset Personalizada](https://www.google.com/search?q=%2312---creating-a-custom-dataset-class)
        * **[Exercício 1 - PlantsDataset](#exercise-1---plantsdataset)**


    * [1.3 - Visão Geral das Imagens no Conjunto de Dados](#13---overview-of-the-images-in-the-dataset)


* [2 - Transformações](#2---transformations)
    * [2.1 - Computando a Média e o Desvio Padrão](#21---computing-mean-and-standard-deviation)
    * [2.2 - Definindo as Transformações](#22---defining-transformations)
        * **[Exercício 2 - get_transformations](#exercise-2---get_transformations)**




* [3 - Carregamento de Dados](#3---data-loading)
    * **[Exercício 3 - get_data_loaders](#exercise-3---get_data_loaders)**

<a name='imports'></a>
## Importar as bibliotecas necessárias

In [5]:
# Importa a biblioteca pandas, frequentemente usada para manipulação e análise de dados tabulares.
import pandas as pd

# Importa a biblioteca principal do PyTorch, utilizada para computação de tensores e aprendizado profundo (deep learning).
import torch

# Importa classes e funções essenciais do PyTorch para manipulação de dados:
# - Dataset: Classe base para criar conjuntos de dados personalizados.
# - Subset: Cria um subconjunto de um Dataset existente.
# - DataLoader: Cria um iterador para carregar os dados em lotes (batches) de forma eficiente.
# - random_split: Divide um dataset aleatoriamente em comprimentos especificados.
from torch.utils.data import Dataset, Subset, DataLoader, random_split

# Importa o módulo de transformações do torchvision, utilizado para pré-processamento e aumento de dados (data augmentation) em imagens.
from torchvision import transforms

# Importa a classe Image da biblioteca PIL (Pillow), usada para abrir, manipular e salvar arquivos de imagem.
from PIL import Image

In [4]:
# Importa a função tqdm do módulo tqdm.auto, que fornece uma barra de progresso visual 
# que se adapta automaticamente ao ambiente de execução (como terminal ou Jupyter Notebook).
from tqdm.auto import tqdm

# Importa o módulo 'helper_utils', que presumivelmente contém funções auxiliares 
# específicas do projeto (como plotagem de gráficos ou carregamento de dados).
import helper_utils_6

<a name='1---data-access'></a>
## 1 - Acesso aos Dados 

<a name='11---exploring-the-dataset'></a>
### 1.1 - Explorando o Conjunto de Dados

Como você já aprendeu, o primeiro passo ao trabalhar com qualquer novo conjunto de dados é explorá-lo. Isso envolve compreender sua estrutura, os tipos de dados que ele contém e identificar quaisquer problemas potenciais, tais como valores ausentes ou discrepantes (*outliers*).

Nesta etapa, você usará a função `print_data_folder_structure` do módulo `helper_utils` para exibir a estrutura de pastas do conjunto de dados.

Isso o ajudará a visualizar como os arquivos e diretórios estão organizados, o que é um passo crucial antes de começar a carregar e pré-processar os dados.

In [ ]:
# Importa a função tqdm do módulo tqdm.auto, que fornece uma barra de progresso visual 
# que se adapta automaticamente ao ambiente de execução (como terminal ou Jupyter Notebook).
from tqdm.auto import tqdm

# Importa o módulo 'helper_utils', que presumivelmente contém funções auxiliares 
# específicas do projeto (como plotagem de gráficos ou carregamento de dados).
import helper_utils

# Importa o módulo 'unittests', provavelmente um arquivo local contendo 
# testes unitários para validar o funcionamento do código deste projeto.
import unittests

plants_dataset/
├── classname.txt
├── df_labels.csv
├── df_labels_orig.csv
├── aloevera/
├── banana/
├── bilimbi/
├── cantaloupe/
├── cassava/
├── coconut/
├── corn/
├── cucumber/
├── curcuma/
├── eggplant/
├── galangal/
├── ginger/
├── guava/
├── kale/
├── longbeans/
├── mango/
├── melon/
├── orange/
├── paddy/
├── papaya/
├── peperchili/
├── pineapple/
├── pomelo/
├── shallot/
├── soybeans/
├── spinach/
├── sweetpotatoes/
├── tobacco/
├── waterapple/
└── watermelon/


Agora você tem uma compreensão inicial da estrutura do conjunto de dados:

* `df_labels.csv`,
* `classname.txt`,
* Uma pasta por classe, cada uma contendo as imagens daquela classe (todas no formato `.jpg`).

Essas informações serão úteis quando você projetar sua classe `Dataset` personalizada mais adiante.

In [7]:
# Imprime o conteúdo do arquivo `df_labels.csv`
# Lê o arquivo CSV localizado no caminho especificado e o carrega em um DataFrame do pandas
df_labels = pd.read_csv(f'{path_dataset}/df_labels.csv')

# Exibe as 5 primeiras linhas do DataFrame para visualizar a estrutura e os dados iniciais
print(df_labels.head())

                 image:FILE  category
0  aloevera/aloevera700.jpg         0
1  aloevera/aloevera701.jpg         0
2  aloevera/aloevera702.jpg         0
3  aloevera/aloevera703.jpg         0
4  aloevera/aloevera704.jpg         0


In [8]:
# Imprime o conteúdo do arquivo `classname.txt`
# Abre o arquivo de texto no modo de leitura ('r'), garantindo seu fechamento automático e adequado após o uso
with open(f'{path_dataset}/classname.txt', 'r') as f:
    # Lê todo o conteúdo do arquivo e o divide em uma lista de strings, removendo as quebras de linha (newlines)
    class_names = f.read().splitlines()

# Exibe a lista contendo os nomes das classes lidas do arquivo
print(class_names)

['aloevera', 'banana', 'bilimbi', 'cantaloupe', 'cassava', 'coconut', 'corn', 'cucumber', 'curcuma', 'eggplant', 'galangal', 'ginger', 'guava', 'kale', 'longbeans', 'mango', 'melon', 'orange', 'paddy', 'papaya', 'peperchili', 'pineapple', 'pomelo', 'shallot', 'soybeans', 'spinach', 'sweetpotatoes', 'tobacco', 'waterapple', 'watermelon']


Você verificou que o arquivo `df_labels.csv` contém os rótulos de cada imagem junto com seus respectivos nomes de arquivo, e que o arquivo `classname.txt` contém os nomes de todas as classes.

<a name='12---creating-a-custom-dataset-class'></a>
### 1.2 - Criando uma Classe Dataset Personalizada

Chegou a hora de criar uma classe *dataset* personalizada para manipular o conjunto de dados de imagens de plantas.
Esta classe herdará de `torch.utils.data.Dataset` e será responsável por carregar e pré-processar as imagens junto com seus respectivos rótulos.

<a name='exercise-1---plantsdataset'></a>
#### **Exercício 1 - `PlantsDataset`**

**Sua Tarefa:**

Sua tarefa é concluir a implementação da classe PyTorch Dataset personalizada `PlantsDataset`.
Você precisa implementar o código que está faltando em múltiplas seções dentro da classe:

* **Conclua o método `__init__**`:
* Carregue os rótulos a partir do DataFrame utilizando o método `load_labels` (já definido) no atributo `.df_info`.
* Crie um mapeamento dos números inteiros dos rótulos para os nomes das classes utilizando o método `read_classname` (já definido).


* **Conclua o método `__len__**`:
* Retorne o número total de amostras no conjunto de dados extraindo o comprimento do atributo `.labels`.


* **Conclua o método `__getitem__**`:
* Recupere a imagem no índice especificado utilizando o método existente `retrieve_image`.
* Aplique as transformações à imagem, caso estejam especificadas.
* Obtenha o rótulo correspondente a partir do atributo `.labels`.

<details>
  <summary><b><font color="green">Dicas de Código Adicionais (Clique para expandir se estiver travado)</font></b></summary>
  
Se precisar de uma ajuda, aqui está um guia mais detalhado para cada método:

**Para o método `__init__`:**

* Para `self.labels`: Chame `self.load_labels()` para extrair os rótulos do DataFrame `self.df_info`.

**Para o método `__len__`:**

* Use a função embutida `len()` em `self.labels`.

**Para o método `__getitem__`:**

* Use `self.retrieve_image(idx)` para obter a imagem no índice especificado.
* Se `self.transform` não for `None`, aplique-o à imagem usando `self.transform(image)`.
* Obtenha o rótulo a partir de `self.labels[idx]`.

</details>

In [9]:
# CLASSE AVALIADA: PlantsDataset
class PlantsDataset(Dataset):
    """
    Uma classe de dataset personalizada para carregar imagens de plantas e seus respectivos rótulos.

    Args:
        root_dir (str): Diretório raiz contendo os arquivos do dataset, incluindo 'classname.txt'.
        transform (callable, opcional): Transformação opcional a ser aplicada em uma amostra.

    Atributos:
        root_dir (str): Caminho para o diretório raiz do dataset.
        transform (callable): Transformações a serem aplicadas às imagens.
        df_info (pd.DataFrame): DataFrame contendo os nomes dos arquivos de imagem e os rótulos de categoria.
        labels (list): Lista de rótulos inteiros para cada imagem.
        class_names (list): Lista de nomes de classes correspondentes aos índices dos rótulos.
    """

    def __init__(self, root_dir, transform=None):
        """
        Inicializa o objeto do dataset.

        Args:
            root_dir (str): Caminho para o diretório raiz contendo o dataset.
            transform (callable, opcional): Transformação opcional a ser aplicada em uma amostra.
        """

        # Inicializa o caminho para o diretório raiz e as transformações
        self.root_dir = root_dir
        self.transform = transform

        # Lê o arquivo CSV (com o caminho das imagens e os rótulos das categorias)
        self.df_info = self.read_df()

        ### INICIE SEU CÓDIGO AQUI ###

        # Carrega os rótulos do DataFrame usando o método `load_labels`
        self.labels = None(None)

        # Cria um mapeamento de inteiros de rótulo para nomes de classes usando o método `read_classname`
        self.class_names = None()

        ### TERMINE SEU CÓDIGO AQUI ###

    def read_df(self):
        """
        Lê um arquivo CSV do caminho especificado e o retorna como um DataFrame do pandas.
        """
        path_csv = self.root_dir + "/df_labels.csv"
        df = pd.read_csv(path_csv)
        return df

    def read_classname(self):
        """
        Lê os nomes das classes de um arquivo chamado 'classname.txt' localizado no diretório raiz.

        Returns:
            list: Uma lista de nomes de classes, cada um como uma string, lida a partir do arquivo.
        """
        path_txt = self.root_dir + "/classname.txt"
        with open(path_txt, "r") as f:
            class_names = f.read().splitlines()
        return class_names

    def load_labels(self, df):
        """
        Extrai os inteiros de rótulo de um DataFrame e os retorna como uma lista.
        """
        labels = []

        for idx, row in df.iterrows():
            label_int = row["category"]
            labels.append(label_int)
        return labels

    def get_label_description(self, label: int):
        """
        Retorna a descrição de um rótulo de classe.
        """
        description = self.class_names[label]
        return description

    def retrieve_image(self, idx: int):
        """
        Recupera e retorna da pasta a imagem PIL no índice especificado.
        Também converte a imagem para o modo RGB.
        """
        img_path = self.root_dir + "/" + self.df_info.iloc[idx]["image:FILE"]
        with Image.open(img_path) as img:
            image = img.convert("RGB")
        return image

    ### INICIE SEU CÓDIGO AQUI ###

    def __len__(self):
        """
        Retorna o número de amostras no dataset.
        """
        # Retorna o número total de amostras a partir do atributo `.labels`
        length = None
        return length

    def __getitem__(self, idx):
        """
        Recupera a imagem e seu respectivo rótulo no índice especificado.

        Args:
            idx (int): Índice do item a ser recuperado.

        Returns:
            tuple: Uma tupla (image, label) onde:
                - image: A imagem no índice fornecido, possivelmente transformada se uma transformação for especificada.
                - label: O rótulo correspondente à imagem.
        """
        # Recupera a imagem usando o método `retrieve_image`
        image = None(None)

        # Aplica as transformações especificadas à imagem, se houver alguma
        # O None da condição do if não faz parte do exercício, deixe como está
        if self.transform is not None:
            image = None(None)

        # Recupera o rótulo a partir do atributo `labels`
        label = None

        # Retorna a imagem e o rótulo
        return image, label  

    ### TERMINE SEU CÓDIGO AQUI ###

<>:37: SyntaxWarning: 'NoneType' object is not callable; perhaps you missed a comma?
<>:40: SyntaxWarning: 'NoneType' object is not callable; perhaps you missed a comma?
<>:115: SyntaxWarning: 'NoneType' object is not callable; perhaps you missed a comma?
<>:120: SyntaxWarning: 'NoneType' object is not callable; perhaps you missed a comma?
/tmp/ipykernel_10251/2969038148.py:37: SyntaxWarning: 'NoneType' object is not callable; perhaps you missed a comma?
  self.labels = None(None)
/tmp/ipykernel_10251/2969038148.py:40: SyntaxWarning: 'NoneType' object is not callable; perhaps you missed a comma?
  self.class_names = None()
/tmp/ipykernel_10251/2969038148.py:115: SyntaxWarning: 'NoneType' object is not callable; perhaps you missed a comma?
  image = None(None)
/tmp/ipykernel_10251/2969038148.py:120: SyntaxWarning: 'NoneType' object is not callable; perhaps you missed a comma?
  image = None(None)


In [ ]:
# Inicializa o dataset personalizado de plantas (PlantsDataset)
# root_dir: especifica o diretório onde os arquivos de dados e imagens estão armazenados
# transform=None: indica que nenhuma transformação (como redimensionamento ou data augmentation) será aplicada às imagens neste momento
plants_dataset = PlantsDataset(root_dir=path_dataset, transform=None)

In [ ]:
# Imprime o tamanho (quantidade total de amostras) do dataset
print(f'Tamanho do dataset: {len(plants_dataset)}')

In [ ]:
# Look at a sample to check it's working correctly
sel_idx = 10
img, label = plants_dataset[sel_idx]

# Visualize the image
helper_utils.plot_img(img)

# Print its description
print(f'Description: {plants_dataset.get_label_description(label)}')

# Print its shape
print(f'Image shape: {img.size}\n')  # PIL image size is (width, height)

##### **Expected Output**
```
Description: aloevera
Image shape: (269, 187)
```

![exp_out_1.png](exp_out_1.png)

In [ ]:
# Test your code!
unittests.exercise_1(PlantsDataset)

<a name='13---overview-of-the-images-in-the-dataset'></a>
### 1.3 - Visão geral das imagens no conjunto de dados

As imagens agora estão acessíveis por meio da classe *dataset* personalizada que você implementou no exercício anterior. No entanto, elas ainda não foram pré-processadas, uma etapa necessária antes de fornecê-las a uma rede neural.

Nesta etapa, você explorará o conjunto de dados utilizando a função `visual_exploration` do módulo `helper_utils_6`.
Esta função exibe algumas imagens de amostra junto com seus respectivos rótulos, permitindo que você inspecione visualmente os dados e tenha uma ideia de suas características principais.

In [ ]:
helper_utils.visual_exploration(plants_dataset, num_rows=2, num_cols=4)

A partir da exploração visual, você pode ver que as imagens no conjunto de dados variam em tamanho, cor e plano de fundo.

Esse tipo de variabilidade é comum em conjuntos de dados do mundo real e ressalta a importância de etapas de pré-processamento — como redimensionamento, normalização e aumento de dados — para ajudar o modelo a generalizar de forma eficaz entre diferentes tipos de imagens.

<a name='2---transformations'></a>
## 2 - Transformações

Antes de fornecer as imagens a uma rede neural, você precisa pré-processá-las utilizando uma série de transformações.
Essas etapas incluem o redimensionamento das imagens para um tamanho consistente, a conversão para tensores e a normalização dos valores dos seus pixels.

<a name='21---computing-mean-and-standard-deviation'></a>

### 2.1 - Computando a Média e o Desvio Padrão

Abaixo está uma função auxiliar, `get_mean_std`, que calcula a média e o desvio padrão do conjunto de dados de treinamento.
Essas estatísticas são necessárias para a etapa de normalização no pipeline de pré-processamento.

Como o redimensionamento e a conversão de imagens em tensores alteram a distribuição dos valores dos pixels, a média e o desvio padrão devem ser computados após a aplicação dessas transformações.

Na função `get_mean_std`, você fará o seguinte:

* **Configuração do Pré-processamento**:
Um pipeline de transformação redimensiona as imagens para 128×128 e as converte para tensores.
* **Primeira Passagem — Computar a Média**:
Para cada imagem, os pixels são achatados (*flattened*) e os valores dos pixels, separados por canal, são somados globalmente em todo o conjunto de dados.
Ao dividir pelo número total de pixels, obtém-se a média por canal.
* **Segunda Passagem — Computar o Desvio Padrão**:
Com a média conhecida, calculamos a diferença ao quadrado entre cada pixel e a média do seu respectivo canal, acumulamos isso por todo o conjunto de dados e, em seguida, calculamos a raiz quadrada para obter o desvio padrão por canal.

**Nota**:
Geralmente, a média e o desvio padrão devem ser computados apenas no conjunto de treinamento.
Usar estatísticas computadas a partir dos dados de teste ou validação pode introduzir vazamento de dados (*data leakage*), no qual informações do conjunto de avaliação influenciam o processo de treinamento.
Neste caso, como os dados ainda não foram divididos, você computará as estatísticas em todo o conjunto de dados por simplicidade.
Os valores de média e desvio padrão que você obterá aqui não mudarão muito quando computados exclusivamente no conjunto de treinamento.

In [ ]:
def get_mean_std(dataset: Dataset):
    # Define the resizing and tensor conversion pipeline
    preprocess = transforms.Compose(
        [transforms.Resize((128, 128)), transforms.ToTensor()]
    )
    
    # Pass 1: Mean Calculation
    total_pixels = 0
    sum_pixels = torch.zeros(3)
    
    # [Visual] Wrap dataset in tqdm to create the progress bar iterator
    mean_loader = tqdm(dataset, desc="Pass 1/2: Computing Mean")
    
    for img, _ in mean_loader:
        # Core computation for mean
        img_tensor = preprocess(img)
        pixels = img_tensor.view(3, -1) # [channels, pixels]
        sum_pixels += pixels.sum(dim=1)
        total_pixels += pixels.size(1)
    
    mean = sum_pixels / total_pixels
    
    # Pass 2: Standard Deviation Calculation
    sum_squared_diff = torch.zeros(3)
    
    # [Visual] Wrap dataset in tqdm to create the progress bar iterator
    std_loader = tqdm(dataset, desc="Pass 2/2: Computing Std")
    
    for img, _ in std_loader:
        # Core computation for std
        img_tensor = preprocess(img)
        pixels = img_tensor.view(3, -1) # [channels, pixels]
        diff = pixels - mean.unsqueeze(1)
        sum_squared_diff += (diff ** 2).sum(dim=1)
    
    std = torch.sqrt(sum_squared_diff / total_pixels)
    
    return mean, std

In [ ]:
# Define the transformations to make to the images
mean, std = get_mean_std(plants_dataset)

print(f"\nMean: {mean}")
print(f" Std: {std}")

<br>
<details>
<summary><b>O Algoritmo Principal (Sem Barra de Progresso)</b></summary>
<br>
A implementação da função acima utiliza o tqdm para fornecer um indicador visual da velocidade de iteração e do tempo estimado restante.

É importante entender que envolver o conjunto de dados em `tqdm(dataset)` não altera os dados ou a matemática. O iterador produz exatamente as mesmas imagens, na exata mesma ordem.

Se você remover a lógica da interface do usuário (UI) para se concentrar estritamente no Algoritmo Matemático, a implementação ficará assim:

```python
def get_mean_std(dataset: Dataset):
    """
    Calcula a média e o desvio padrão dos pixels em um dataset de imagens.

    Args:
        dataset (Dataset): O conjunto de dados contendo as imagens originais.

    Returns:
        mean (torch.Tensor): Um tensor contendo a média para cada canal de cor (RGB).
        std (torch.Tensor): Um tensor contendo o desvio padrão para cada canal de cor (RGB).
    """
    # Define um pré-processamento básico para padronizar o tamanho das imagens e convertê-las para tensores.
    preprocess = transforms.Compose(
        [transforms.Resize((128, 128)), transforms.ToTensor()]
    )
    
    # Passagem 1: Cálculo da Média
    # Variável para armazenar o número total de pixels processados.
    total_pixels = 0
    # Tensor zerado de tamanho 3 (para os canais R, G e B) para acumular a soma dos pixels.
    sum_pixels = torch.zeros(3)
    
    # Iterar diretamente sobre o conjunto de dados sem o encapsulador (wrapper) visual
    for img, _ in dataset:
        # Aplica o pré-processamento na imagem atual.
        img_tensor = preprocess(img)
        # Redimensiona o tensor para agrupar todos os pixels por canal: formato [canais, pixels]
        pixels = img_tensor.view(3, -1)  
        # Acumula a soma dos valores dos pixels ao longo da dimensão dos pixels (dim=1).
        sum_pixels += pixels.sum(dim=1)
        # Incrementa a contagem de pixels totais lidos a partir desta imagem.
        total_pixels += pixels.size(1)
```
    
    

<a name='22---defining-transformations'></a>
### 2.2 - Defining Transformations

Having computed the mean and standard deviation of the dataset, you can now define the transformations to apply to the images.
You’ll create two sets of transformations: one for the training set, which includes data augmentation, and another for the validation and test sets.

<a name='exercise-2---get_transformations'></a>
#### **Exercício 2 - `get_transformations`**

**Sua Tarefa:**

Sua tarefa é implementar o código ausente na função `get_transformations` para criar dois pipelines de transformação de imagens para o PyTorch.

Você implementará as seguintes seções:

* **Definir `main_tfs`**:
* Criar uma transformação `Resize` para redimensionar as imagens para 128x128 pixels.
* Criar uma transformação `ToTensor` para converter imagens PIL em tensores do PyTorch.
* Criar uma transformação `Normalize` usando os valores fornecidos de média e desvio padrão.


* **Definir `augmentation_tfs**`:
* Criar uma transformação `RandomVerticalFlip` com 50% de probabilidade.
* Criar uma transformação `RandomRotation` que rotaciona as imagens em ±15 graus.


* **Compor Pipelines de Transformação**:
* Criar `main_transform` combinando as transformações principais em um único pipeline usando `transforms.Compose`.
* Criar `transform_with_augmentation` combinando tanto as transformações de aumento (augmentation) quanto as principais em um pipeline aumentado.
As transformações de aumento devem ser aplicadas antes das transformações principais.


<details>
  <summary><b><font color="green">Dicas Adicionais de Código (Clique para expandir se estiver travado)</font></b></summary>

Se você precisar de uma ajuda, aqui está um guia mais detalhado para cada seção:

**Para `main_tfs`:**

* Para `Resize`: Use `transforms.Resize((128, 128))` para redimensionar todas as imagens para 128x128 pixels.
* Para `ToTensor`: Use `transforms.ToTensor()` para converter imagens PIL em tensores do PyTorch.
* Para `Normalize`: Use `transforms.Normalize(mean=mean, std=std)` com os parâmetros de média (mean) e desvio padrão (std) fornecidos.

**Para `augmentation_tfs`:**

* Para `RandomVerticalFlip`: Use `transforms.RandomVerticalFlip(p=0.5)` para espelhar as imagens verticalmente com 50% de probabilidade.
* Para `RandomRotation`: Use `transforms.RandomRotation(degrees=15)` para rotacionar as imagens aleatoriamente em um intervalo de ±15 graus.

**Para compor as transformações:**

* Para `main_transform`: Use `transforms.Compose(main_tfs)` para combinar a lista de transformações principais.
* Para `transform_with_augmentation`: Use `transforms.Compose(augmentation_tfs + main_tfs)` para combinar ambas as listas.

</details>

In [ ]:
# GRADED FUNCTION : get_transformations
def get_transformations(mean, std):
    """
    Returns two sets of image transformation pipelines: one with basic preprocessing and another with additional data augmentation.

    Args:
        mean: Sequence of mean values for normalization.
        std: Sequence of standard deviation values for normalization.

    Returns:
        main_transform: Transformation pipeline with resizing, tensor conversion, and normalization.
        transform_with_augmentation: Transformation pipeline including random vertical flip, random rotation, resizing, tensor conversion, and normalization.
    """
    ### START CODE HERE ###
    main_tfs = [  
        # Resize images to 128x128 pixels
        None.None()
        # Convert images to PyTorch tensors
        None.None()
        # Normalize images using the provided mean and std
        None.None()
    ]  

    augmentation_tfs = [  
        # Randomly flip the image vertically
        None.None()
        # Randomly rotate the image by ±15 degrees
        None.None()
    ]  

    # Compose the main transformations into a single pipeline
    main_transform = None.None(None)

    transform_with_augmentation = None.None(None + None)

    ### END CODE HERE ###
    return main_transform, transform_with_augmentation

In [ ]:
# Get the transformations
main_transform, transform_with_augmentation = get_transformations(mean, std)

# Print the transformations to verify
print(main_transform)
print(transform_with_augmentation)

##### **Saída Esperada**

```
Compose(
    Resize(size=(128, 128), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=tensor([0.6659, 0.6203, 0.4784]), std=tensor([0.2888, 0.2884, 0.3426]))
)
Compose(
    RandomVerticalFlip(p=0.5)
    RandomRotation(degrees=[-15.0, 15.0], interpolation=nearest, expand=False, fill=0)
    Resize(size=(128, 128), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=tensor([0.6659, 0.6203, 0.4784]), std=tensor([0.2888, 0.2884, 0.3426]))
)
```

You can verify your transformations by applying them to a sample image from the dataset and inspecting the result.

In [ ]:
# Check main_transform on a sample image
img_transformed = main_transform(img)
print(f"Transformed Image shape: {img_transformed.shape}\n")


# get denormalization function
denormalize = helper_utils.Denormalize(mean, std)
# visualize the augmented image
img_augmented = transform_with_augmentation(img)
helper_utils.plot_img(denormalize(img_augmented), info=f"Augmented Image")

<a name='3---data-loading'></a>
## 3 - Carregamento de Dados

Com sua classe de dataset personalizada e transformações definidas, agora você pode criar carregadores de dados (*data loaders*) para carregar e agrupar os dados em lotes (*batches*) de forma eficiente para treinamento e avaliação. Este é o último passo antes de treinar uma rede neural neste dataset.

Assim como no laboratório anterior, após usar o `random_split` para dividir o dataset em conjuntos de treino, validação e teste, você precisa garantir que cada subconjunto utilize as transformações adequadas.
Uma maneira de alterar as transformações de cada subconjunto é encapsulando-os em novas instâncias da classe de dataset personalizada `SubsetWithTransform`.

In [ ]:
class SubsetWithTransform(Dataset):
    """A subset of a dataset with a specific transform applied."""

    def __init__(self, subset: Subset, transform=None):
        # subset should be a subset WITHOUT transform
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        image, label = self.subset[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

<a name='exercise-3---get_data_loaders'></a>
#### **Exercício 3 - `get_data_loaders`**

**Sua Tarefa:**

Sua tarefa é concluir a implementação da função `get_dataloaders` para dividir um dataset e criar DataLoaders do PyTorch para treinamento, validação e teste.
Você precisa implementar o código ausente em três seções principais:

* **Dividir o Dataset**:
* Use o `random_split` para dividir o dataset em conjuntos de treino, validação e teste com base nos tamanhos calculados.


* **Aplicar Transformações a Cada Subconjunto**:
* Encapsule cada divisão (split) do dataset com `SubsetWithTransform` para aplicar as transformações adequadas.
* Use `augmentation_transform` para o conjunto de treino para incluir o aumento de dados (data augmentation).
* Use `main_transform` para os conjuntos de validação e teste (sem necessidade de aumento de dados).


* **Criar os DataLoaders**:
* Crie objetos `DataLoader` para cada subconjunto do dataset com o tamanho de lote (batch size) especificado.
* Habilite o embaralhamento (shuffling) no carregador de treino para randomizar a ordem dos lotes.
* Desabilite o embaralhamento nos carregadores de validação e teste para manter a consistência na avaliação.

<details>
  <summary><b><font color="green">Dicas Adicionais de Código (Clique para expandir se estiver travado)</font></b></summary>
  
Se você precisar de uma ajuda, aqui está um guia mais detalhado para cada seção:

**Para dividir o dataset:**

* Use `random_split(dataset, [train_size, val_size, test_size])` para dividir o dataset.

**Para aplicar as transformações:**

* Use `SubsetWithTransform(dataset_split, transform=transform_to_apply)` para cada divisão (subconjunto).

**Para criar os DataLoaders:**

* Use `DataLoader(dataset=dataset_split, batch_size=batch_size, shuffle=shuffle_setting)`.
* Para o carregador de treino: defina `shuffle=True` para randomizar a ordem dos lotes.
* Para os carregadores de validação e teste: defina `shuffle=False` para manter a ordem consistente para a avaliação.
* Todos os carregadores devem usar o mesmo parâmetro `batch_size`.

</details>

In [ ]:
# GRADED FUNCTION : get_dataloaders
def get_dataloaders(
    dataset,
    batch_size,
    val_fraction,
    test_fraction,
    main_transform,
    augmentation_transform,
):
    """
    Splits a dataset into training, validation, and test sets, applies specified transforms to each split,
    and returns corresponding DataLoader objects.

    Args:
        dataset: The full dataset to be split.
        batch_size: Number of samples per batch to load.
        val_fraction: Fraction of the dataset to use for validation.
        test_fraction: Fraction of the dataset to use for testing.
        main_transform: Transform to apply to validation and test splits.
        augmentation_transform: Transform to apply to the training split.

    Returns:
        train_loader: DataLoader for the training set with augmentation transforms.
        val_loader: DataLoader for the validation set with main transforms.
        test_loader: DataLoader for the test set with main transforms.
    """

    # Calculate the sizes of each split
    total_size = len(dataset)
    val_size = int(total_size * val_fraction)
    test_size = int(total_size * test_fraction)
    train_size = total_size - val_size - test_size

    ### START CODE HERE ###

    # Split the dataset into training, validation, and test sets
    train_dataset, val_dataset, test_dataset = None(
            None, None
    )  

    # Create dataset with the corresponding transforms for each split
    train_dataset = None(None, None)
    val_dataset = None(None, None)
    test_dataset = None(None, None)

    # Create DataLoaders for each split
    train_loader = None(None)
    val_loader = None(None)
    test_loader = None(None)

    ### END CODE HERE ###
    return train_loader, val_loader, test_loader

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders(
    dataset=plants_dataset,
    batch_size=32,
    val_fraction=0.15,
    test_fraction=0.2,
    main_transform=main_transform,
    augmentation_transform=transform_with_augmentation,
)

In [ ]:
print('=== Train Loader ===')
print(f"Number of batches in train_loader: {len(train_loader)}")
train_dataset = train_loader.dataset
print(f"Number of samples in train_dataset: {len(train_dataset)}")
print(f"Transforms applied to train_dataset: {train_dataset.transform}")
print(f"train_dataset type: {type(train_dataset)}")

print('\n=== Test Loader ===')
print(f"Number of batches in test_loader: {len(test_loader)}")
test_dataset = test_loader.dataset
print(f"Number of samples in test_dataset: {len(test_dataset)}")
print(f"Transforms applied to test_dataset: {test_dataset.transform}")
print(f"test_dataset type: {type(test_dataset)}")

##### **Saída Esperada**

```
=== Train Loader ===
Number of batches in train_loader: 61
Number of samples in train_dataset: 1950
Transforms applied to train_dataset: Compose(
    RandomVerticalFlip(p=0.5)
    RandomRotation(degrees=[-15.0, 15.0], interpolation=nearest, expand=False, fill=0)
    Resize(size=(128, 128), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=tensor([0.6659, 0.6203, 0.4784]), std=tensor([0.2888, 0.2884, 0.3426]))
)
train_dataset type: <class '__main__.SubsetWithTransform'>

=== Test Loader ===
Number of batches in test_loader: 19
Number of samples in test_dataset: 600
Transforms applied to test_dataset: Compose(
    Resize(size=(128, 128), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=tensor([0.6659, 0.6203, 0.4784]), std=tensor([0.2888, 0.2884, 0.3426]))
)
test_dataset type: <class '__main__.SubsetWithTransform'>
```

## Conclusão

Agora você construiu um pipeline de dados de ponta a ponta no PyTorch.

Neste laboratório, você aprendeu como construir um pipeline completo para lidar com um dataset de imagens do mundo real. Você foi além do carregamento básico de dados para explorar os componentes principais que formam um pipeline de dados do PyTorch.

Você criou um **`Dataset`** personalizado para **acesso aos dados**, definiu uma sequência de **transformações** como redimensionamento, normalização e aumento de dados (*data augmentation*) para melhorar a robustez do treinamento, dividiu o dataset em conjuntos de treino, validação e teste, e usou o **`DataLoader`** para iteração e agrupamento em lotes (*batching*) de forma eficiente.

Com esses componentes fundamentais — `Dataset`, `Transforms` e `DataLoader` — agora você tem um fluxo de trabalho limpo, eficiente e reutilizável para preparar qualquer dataset de imagens para o treinamento de uma rede neural.